# Projet de Scraping Booking.com
Nous allons construire ensemble un **robot (spider)** capable de collecter des données sur les hôtels Booking.com. C



## 1. Importation des Librairies

Nous commençons par importer les outils nécessaires. 
- **Scrapy** est notre moteur de scraping.
- **re** (Regular Expressions) nous servira à nettoyer les textes (ex: enlever le symbole € du prix).
- **os** nous permet d'interagir avec le système de fichiers (vérifier si un fichier existe).

In [1]:
import scrapy
from scrapy.crawler import CrawlerProcess
import re
import os

# scrapy : Le framework principal pour le scraping web.
# CrawlerProcess : Permet de lancer le spider directement depuis ce script Python/Notebook sans passer par la ligne de commande.
# re : Module pour les expressions régulières 
# os : Module pour interagir avec le système d'exploitation (gestion des fichiers).

## 2. Gestion des Cookies (Contournement des protections)

Les sites comme Booking.com détectent facilement les robots. Une technique pour passer inaperçu est d'utiliser un **Cookie** valide, récupéré depuis votre propre navigateur.

Cette fonction charge le contenu de votre fichier `cookie.txt` pour l'injecter dans nos requêtes. Ici le choix a été fait de garder le cookie dans un fichier txt afin de ne pas casser le code accidentellement en raison de sa longueur et de pouvoir le modifier facilement si jamais il venait à se périmer.

In [2]:
# --- FONCTION DE CHARGEMENT DU COOKIE ---
def load_cookie_from_file(filename="cookie.txt"):
    """
    Charge le cookie brut depuis un fichier texte.
    Retourne la chaîne de caractères du cookie ou None si le fichier est absent.
    """
    # Vérifie si le fichier existe pour éviter de faire planter le programme
    if not os.path.exists(filename):
        print(f" ERREUR : Le fichier '{filename}' n'existe pas !")
        return None
    
    # Ouvre le fichier en lecture ('r') avec l'encodage utf-8
    with open(filename, 'r', encoding='utf-8') as f:
        # Lit le contenu et nettoie les sauts de ligne inutiles
        return f.read().replace('\n', '').replace('\r', '').strip()

# Chargement du cookie dans une variable globale
MY_COOKIE = load_cookie_from_file()

## 3. Création de la Classe Spider

Dans Scrapy, un **Spider** est une classe qui définit comment naviguer sur un site. 
Ici, nous définissons la structure de base.

In [3]:
class BookingDeepSpider(scrapy.Spider):
    # Nom unique du spider, utilisé par Scrapy pour l'identifier
    name = "booking_deep_v2"
    
    # Liste des domaines autorisés. Le spider ne sortira pas de ce site.
    allowed_domains = ["booking.com"]

## 4. Définition des Cibles (Villes)

Nous définissons ici la liste des villes que nous souhaitons scrapper.

In [4]:
# On attache la liste des villes à notre classe Spider
BookingDeepSpider.cities = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen", "Paris", 
    "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg", "Colmar", 
    "Eguisheim", "Besancon", "Dijon", "Annecy", "Grenoble", "Lyon", 
    "Gorges du Verdon", "Bormes les Mimosas", "Cassis", "Marseille", 
    "Aix en Provence", "Avignon", "Uzes", "Nimes", "Aigues Mortes", 
    "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege", 
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle"
]

## 5. Configuration Avancée (Settings)

Nous configurons :
- **USER_AGENT** : Pour se faire passer pour un navigateur Chrome standard.
- **DOWNLOAD_DELAY** : Pour attendre 1.5 seconde entre chaque requête (politesse et discrétion).
- **FEEDS** : Pour dire à Scrapy d'enregistrer automatiquement les résultats dans un fichier CSV.

In [5]:
BookingDeepSpider.custom_settings = {
    # Identité du navigateur simulé (Chrome sur Windows 10)
    'USER_AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    
    # Ne pas respecter le fichier robots.txt (souvent restrictif sur les sites commerciaux)
    'ROBOTSTXT_OBEY': False,
    
    # Pause de 1.5 secondes entre chaque requête pour ne pas surcharger le serveur
    'DOWNLOAD_DELAY': 1.5,
    
    # On désactive la gestion automatique des cookies par Scrapy car on l'injecte manuellement
    'COOKIES_ENABLED': False,
    
    # En-têtes HTTP par défaut pour ressembler à une vraie requête navigateur
    'DEFAULT_REQUEST_HEADERS': {
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7',
        'Cookie': MY_COOKIE, # Injection de notre cookie magique
    },
    
    # Configuration de l'export des données
    'FEEDS': {
        'booking_full_data.csv': {
            'format': 'csv', 
            'encoding': 'utf8', 
            'overwrite': True # Écrase le fichier s'il existe déjà
        },
    },
    
    # Niveau de log pour voir ce qui se passe 
    'LOG_LEVEL': 'INFO'
}

## 6. Démarrage des Requêtes (start_requests)

La méthode `start_requests` est le point d'entrée du Spider. Elle génère la première URL pour chaque ville.

**Note pédagogique** : Nous utilisons `yield` au lieu de `return`. Cela fait de cette fonction un **générateur**, capable de produire des requêtes une par une sans tout charger en mémoire.

In [6]:
def start_requests(self):
    # Si pas de cookie, on ne lance rien pour éviter d'être bloqué immédiatement
    if not MY_COOKIE: return
    
    base_url = "https://www.booking.com/searchresults.fr.html"
    
    # Dates de test (Juin 2026)
    checkin = "2026-06-06"
    checkout = "2026-06-07"
    
    # On boucle sur chaque ville avec enumerate pour avoir un ID unique (1, 2, 3...)
    for i, city in enumerate(self.cities, 1):
        # Construction de l'URL avec les paramètres 
        url = f"{base_url}?ss={city}&checkin={checkin}&checkout={checkout}&group_adults=1&no_rooms=1&group_children=0"
        
        # On envoie la requête.
        yield scrapy.Request(url=url, callback=self.parse, cb_kwargs={'city': city, 'id': i})

# On attache la méthode à la classe
BookingDeepSpider.start_requests = start_requests

## 7. Analyse de la Liste des Hôtels (parse)

Cette méthode reçoit la page de résultats de recherche. Son rôle est de :
1. Vérifier si on est bloqué.
2. Trouver toutes les "cartes" d'hôtels.
3. Extraire les infos de base (Prix, Note, Nom).
4. **Lancer une nouvelle requête** vers la page de l'hôtel pour avoir la description.

In [7]:
def parse(self, response, city, id):
    # Vérification de sécurité : si l'URL contient "robot" ou "sign-in", c'est mauvais signe
    if "robot" in response.url or "sign-in" in response.url:
        self.logger.error(f" BLOQUÉ sur {city} (Cookie HS).")
        return

    # Sélection de toutes les cartes d'hôtels via leur attribut data-testid
    cards = response.css('div[data-testid="property-card"]')
    
    # On limite à 20 hôtels par ville pour l'exercice
    for card in cards[:20]:
        # Extraction du prix brut
        raw_price = card.css('span[data-testid="price-and-discounted-price"]::text').get()
        # Nettoyage : on ne garde que les chiffres (ex: "120 €" -> "120")
        price = re.sub(r'[^\d]', '', raw_price) if raw_price else "0"
        
        # Extraction de la note (ex: "8,5")
        raw_score = card.css('div[data-testid="review-score"] div::text').get()
        # Remplacement de la virgule par un point pour le format numérique standard
        score = raw_score.replace(',', '.') if raw_score else "0"

        # Extraction du nom de l'hôtel
        name = card.css('div[data-testid="title"]::text').get()
        
        # Extraction de l'URL de la page détail
        url_part = card.css('a[data-testid="title-link"]::attr(href)').get()
        # Reconstruction de l'URL complète si nécessaire
        full_url = url_part if url_part and "http" in url_part else "https://www.booking.com" + (url_part or "")

        # Création d'un dictionnaire temporaire avec les données collectées
        item = {
            'id_ville': id,
            'ville': city,
            'nom_hotel': name,
            'url': full_url,
            'score': score,
            'prix': price
        }

        # On suit le lien pour aller chercher la description (Détails)
        # meta={'item': item} permet de passer les données déjà récoltées à la prochaine étape
        yield scrapy.Request(url=full_url, callback=self.parse_details, meta={'item': item})

# On attache la méthode à la classe
BookingDeepSpider.parse = parse

## 8. Extraction de la Description

Sur la page de l'hôtel, l'information est parfois cachée ou structurée différemment.  i la méthode 1 échoue, on tente la 2, puis la 3. Cette méthode est utilisée en raison d'echecs fréquents sur la description lors des premières versions.

In [8]:
def parse_details(self, response):
    # On récupère l'item (données partielles) passé depuis la fonction précédente via 'meta'
    item = response.meta['item']
    
    # --- STRATÉGIE ROBUSTE D'EXTRACTION ---
    
    # 1. Essai Selecteur Moderne
    desc_parts = response.css('div[data-testid="property-description"] ::text').getall()
    
    # 2. Essai Selecteur Ancien (Fallback pour les vieilles pages)
    if not desc_parts:
        desc_parts = response.css('div#property_description_content ::text').getall()
        
    # 3. Essai Ultime (La balise Meta Description cachée dans le <head>)
    if not desc_parts:
        meta_desc = response.css('meta[name="description"]::attr(content)').get()
        desc_parts = [meta_desc] if meta_desc else []

    # Traitement du texte trouvé
    if desc_parts:
        # On joint tous les morceaux de texte et on nettoie les espaces
        full_text = " ".join([text.strip() for text in desc_parts if text.strip()])
        # On tronque à 500 caractères pour ne pas avoir des textes trop longs
        item['description'] = full_text[:500] + "..." 
    else:
        item['description'] = "Description introuvable (Bloqué ?)"

    #On livre l'item final ! Via Yield
    yield item

# On attache la méthode à la classe
BookingDeepSpider.parse_details = parse_details #rajoute la fonction "feature" parse_details à la classe BookingDeepSpider

## 9. Exécution du Spider

 Il ne reste plus qu'à lancer le processus. 
Le `CrawlerProcess` va gérer l'orchestration, les files d'attente et l'export CSV.

In [9]:
# On utilise multiprocessing pour éviter le plantage du Kernel (problème de reactor Twisted)
from multiprocessing import Process

def run_spider():
    # Création du processus de Scraping
    process = CrawlerProcess()
    
    # Ajout du spider
    process.crawl(BookingDeepSpider)
    
    # Démarrage
    process.start()

# Vérification finale et exécution isolée
if MY_COOKIE:
    print(" Lancement du scraping (isolé dans un processus séparé pour éviter le crash)...")
    p = Process(target=run_spider)
    p.start()
    p.join() # On attend la fin pour ne pas rendre la main trop tôt
    print(" Scraping terminé avec succès !")
else:
    print(" Pas de cookie chargé. Vérifiez votre fichier cookie.txt.")

 Lancement du scraping (isolé dans un processus séparé pour éviter le crash)...


2026-04-11 17:27:58 [scrapy.utils.log] INFO: Scrapy 2.15.0 started (bot: scrapybot)
2026-04-11 17:27:58 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.0.3',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '25.5.0',
 'Python': '3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]',
 'pyOpenSSL': '26.0.0 (OpenSSL 3.5.6 7 Apr 2026)',
 'cryptography': '46.0.7',
 'Platform': 'Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39'}
2026-04-11 17:27:58 [scrapy.crawler] DEBUG: Using CrawlerProcess
2026-04-11 17:27:58 [scrapy.addons] INFO: Enabled addons:
[]
2026-04-11 17:27:58 [scrapy.extensions.telnet] INFO: Telnet Password: bc88fcc11e641bb5
2026-04-11 17:27:58 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.logcount.LogCount',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logs

 Scraping terminé avec succès !


##  Glossaire Technique & Commandes

Voici un récapitulatif des outils et concepts utilisés dans ce projet.

| Fonction / Commande | Paramètres Clés | Définition Simple |
| :--- | :--- | :--- |
| **`scrapy.Spider`** | `name`, `allowed_domains` | La classe de base qui définit un robot. C'est le "cerveau" qui décide où aller. |
| **`scrapy.Request`** | `url`, `callback`, `meta` | Une demande faite à un site web. On lui dit quelle URL visiter et quelle fonction appeler ensuite. |
| **`response.css`** | `selecteur` (ex: `div.class`) | Permet de cibler des éléments HTML précis (comme le titre ou le prix) pour extraire leur contenu. |
| **`yield`** | *aucun* | Mot-clé magique qui permet de "donner" une valeur (un hôtel, une requête) sans arrêter la fonction. |
| **`re.sub`** | `pattern`, `replacement`, `string` | Fonction de "Chercher et Remplacer" puissante. Utilisée ici pour nettoyer le prix (enlever les lettres). |
| **`os.path.exists`** | `path` | Vérifie simplement si un fichier est présent sur l'ordinateur. |
| **`enumerate`** | `iterable`, `start` | Transforme une liste en une suite de paires (index, valeur). Utile pour numéroter les villes. |
| **`cb_kwargs`** | *dictionnaire* | "Callback Keyword Arguments". Permet de passer des infos (comme le nom de la ville) à la fonction suivante. |
| **`meta`** | *dictionnaire* | Le sac à dos du robot. Permet de transporter des données (l'item en cours de construction) d'une page à l'autre. |
| **`CrawlerProcess`** | *settings* | Le chef d'orchestre. Il lance les spiders et gère les configurations techniques (export CSV, délais). |

Donnée	Sélecteur CSS utilisé	Explication
La Carte Hôtel	div[data-testid="property-card"]	C'est la boîte qui contient toutes les infos d'un hôtel.
Nom de l'hôtel	div[data-testid="title"]::text	Le titre en gras.
Prix	span[data-testid="price-and-discounted-price"]::text	Le prix final affiché (ex: "€ 120").
Note	div[data-testid="review-score"] div::text	La note (ex: "8,5"). On vise le div à l'intérieur du bloc score.
Lien (URL)	a[data-testid="title-link"]::attr(href)	L'adresse web pour aller sur la page détail.
Desc. courte	div[data-testid="recommended-units"]::text	Le petit texte type "Chambre Double - Vue Mer".

Priorité Sélecteur CSS utilisé Pourquoi ?Choix 1 (Moderne)
div[data-testid="property-description"] ::textLe nouveau standard de Booking.Choix 2 (Classique)div#property_description_content ::textL'ancien ID standard, encore très présent.Choix 3 (Secours)meta[name="description"]::attr(content)La balise cachée pour Google. Elle contient toujours un résumé fiable si le reste échoue.